Fixed SUM(BOOLEAN) error by using IFF() to cast NULL checks to integers
*Co-authored with CoCo*

# Notebook for Assignment

## Task 1

In [ ]:
%%sql -r dataframe_2
-- create db, schemas, and warehouse 
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE DATABASE ASSIGNMENT_DB;

CREATE OR REPLACE SCHEMA ASSIGNMENT_DB.RAW_SCHEMA;

CREATE OR REPLACE SCHEMA ASSIGNMENT_DB.CLEAN_SCHEMA;

CREATE OR REPLACE WAREHOUSE ASSIGNMENT_WH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 300
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE ASSIGNMENT_WH;
USE DATABASE ASSIGNMENT_DB;
USE SCHEMA RAW_SCHEMA;

## Tasks 2-4

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE STAGE tastybytes_stage
  URL = 's3://sfquickstarts/tastybytes/';

In [ ]:
%%sql -r dataframe_3
LIST @tastybytes_stage/raw_pos/menu/;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE FILE FORMAT menu_ff
  TYPE = CSV
  COMPRESSION = AUTO          -- handles .gz automatically
  FIELD_DELIMITER = ','
  SKIP_HEADER = 0             -- first row is column names
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  TRIM_SPACE = TRUE
  ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
  NULL_IF = ('NULL', 'null', '');

In [ ]:
%%sql -r dataframe_6
LIST @tastybytes_stage/raw_pos/menu/;

SELECT $1, $2, $3, $4, $5, $6, $7, $8, $9, $10, $11
FROM @tastybytes_stage/raw_pos/menu/
LIMIT 100;

## Task 5

In [ ]:
-- create table
CREATE OR REPLACE TABLE raw_menu (
    menu_id NUMBER(19, 0),
    menu_type_id NUMBER(19, 0),
    menu_type VARCHAR,
    truck_brand_name VARCHAR,
    menu_item_id NUMBER (19, 0),
    menu_item_name VARCHAR,
    category VARCHAR,
    subcategory VARCHAR,
    cost_price NUMBER(38, 2),
    sell_price NUMBER(38, 2),
    health_metrics VARIANT
);

In [ ]:
%%sql -r dataframe_7
-- load from stage
COPY INTO raw_menu
FROM @tastybytes_stage/raw_pos/menu/
FILE_FORMAT = (FORMAT_NAME = menu_ff);

## Task 6

In [ ]:
-- number of menu items
SELECT 
    COUNT(DISTINCT(menu_item_name)) AS menu_items,
    COUNT(DISTINCT(truck_brand_name)) AS truck_brands,
    COUNT(DISTINCT(category)) AS categories,
    MIN(sell_price) AS min_sell_price,
    MAX(sell_price) AS max_sell_price,
    ROUND(AVG(sell_price), 2) AS avg_sell_price
FROM raw_menu;

## Task 7

In [ ]:
CREATE OR REPLACE TABLE ASSIGNMENT_DB.CLEAN_SCHEMA.clean_menu AS
SELECT 
    menu_id,
    menu_type_id,
    TRIM(menu_item_name) AS menu_item_name,
    UPPER(truck_brand_name) AS truck_brand_name,
    UPPER(category) AS category,
    sell_price,
    cost_price,
    sell_price - cost_price AS profit,
    ROUND(((sell_price - cost_price) / NULLIF(sell_price, 0)) * 100) AS profit_pct,
    CASE
        WHEN sell_price < 5 THEN 'BUDGET'
        WHEN sell_price < 10 THEN 'STANDARD'
        WHEN sell_price < 15 THEN 'PREMIUM'
        ELSE 'LUXURY'
    END AS price_category,
    UPPER(truck_brand_name) || '-' || TRIM(menu_item_name) AS display_name,
    CURRENT_TIMESTAMP() AS load_timestamp
FROM ASSIGNMENT_DB.RAW_SCHEMA.RAW_MENU;
    

## Task 8

In [ ]:
-- 5 most expensive menu items
SELECT *
FROM ASSIGNMENT_DB.CLEAN_SCHEMA.CLEAN_MENU
ORDER BY sell_price DESC
LIMIT 5;

In [ ]:
-- 5 most profitable menu items
SELECT *
FROM ASSIGNMENT_DB.CLEAN_SCHEMA.CLEAN_MENU
ORDER BY profit DESC
LIMIT 5;

In [ ]:
-- truck with highest avg sell price
SELECT 
    truck_brand_name,
    ROUND(AVG(sell_price), 2) AS avg_sell_price
FROM ASSIGNMENT_DB.CLEAN_SCHEMA.CLEAN_MENU
GROUP BY 1
ORDER BY 2 DESC
LIMIT 1;

In [ ]:
SELECT 
    truck_brand_name,
    ROUND(AVG(profit), 2) AS avg_profit
FROM ASSIGNMENT_DB.CLEAN_SCHEMA.CLEAN_MENU
GROUP BY 1
ORDER BY 2 DESC
LIMIT 1;

In [ ]:
SELECT
    category,
    COUNT(*) AS menu_items
FROM ASSIGNMENT_DB.CLEAN_SCHEMA.CLEAN_MENU
GROUP BY category;

In [ ]:
SELECT
    price_category,
    COUNT(*) AS menu_items
FROM ASSIGNMENT_DB.CLEAN_SCHEMA.CLEAN_MENU
GROUP BY price_category;

## Task 9

In [ ]:
-- Missing-value counts (Task 9 first part)
SELECT
  COUNT_IF(menu_item_name IS NULL OR TRIM(menu_item_name) = '') AS missing_item_name,
  COUNT_IF(category IS NULL OR TRIM(category) = '') AS missing_category,
  COUNT_IF(cost_price IS NULL) AS missing_cost,
  COUNT_IF(sell_price IS NULL) AS missing_sale_price
FROM ASSIGNMENT_DB.CLEAN_SCHEMA.clean_menu;

-- Add quality_status to the clean table (or recreate CTAS with this column)
CREATE OR REPLACE TABLE ASSIGNMENT_DB.CLEAN_SCHEMA.clean_menu AS
SELECT
  c.*,
  CASE
    WHEN missing_cnt = 0 THEN 'GOOD'
    WHEN missing_cnt = 1 THEN 'REVIEW'
    ELSE 'POOR'   -- 2+ missing
  END AS quality_status
FROM (
  SELECT
    *,
    IFF(menu_item_name IS NULL OR TRIM(menu_item_name) = '', 1, 0)
      + IFF(category IS NULL OR TRIM(category) = '', 1, 0)
      + IFF(cost_price IS NULL, 1, 0)
      + IFF(sell_price IS NULL, 1, 0) AS missing_cnt
  FROM ASSIGNMENT_DB.CLEAN_SCHEMA.clean_menu
) c;

## Task 10

In [ ]:
%%sql -r dataframe_18
SELECT
    truck_brand_name,
    COUNT(*) AS total_menu_items,
    ROUND(AVG(sell_price), 2) AS avg_sell_price,
    ROUND(AVG(profit), 2) AS avg_profit,
    MIN(sell_price) AS min_sell_price,
    MAX(sell_price) AS max_sell_price
FROM ASSIGNMENT_DB.CLEAN_SCHEMA.CLEAN_MENU
GROUP BY truck_brand_name
ORDER BY avg_profit DESC;